In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from collections import defaultdict
import numpy as np

import matplotlib.pyplot as plt

import yaml
from pathlib import Path

from thesis_project.datasets import SpeechCommandsGoogle
from thesis_project.datasets.speech_commands.config import ALL_KEYWORDS, CANONICAL, COMMAND_KEYWORDS_20, COMMAND_KEYWORDS_24
from thesis_project.utils.paths import get_data_dir

from thesis_project.models.architectures.keyword_spotting import KeyWordSpottingModel, KeyWordSpottingConfig, SpectrogramConfig, BackboneConfig, NoiseConfig, DatasetConfig



In [18]:
KEY_WORDS_LIST = CANONICAL  # COMMAND_KEYWORDS_20  # COMMAND_KEYWORDS_24  # ALL_KEYWORDS  # CANONICAL

In [19]:
dataset_cfg=DatasetConfig(
    key_words=KEY_WORDS_LIST,
    use_unknown=True,
    use_silence=True,
    upsample=False,
)

noise_eval_cfg=NoiseConfig(
    add_noise=True,
    noise_prob=0.8,
    snr=float("inf"))

In [20]:
from copy import deepcopy
import torch

def make_val_sets(data_dir, noise_eval_cfg):
    cfg = noise_eval_cfg.model_dump()
    val1 = SpeechCommandsGoogle(
        root=str(data_dir),
        subset="validation",
        download=True,
        **cfg,
    )
    val2 = SpeechCommandsGoogle(
        root=str(data_dir),
        subset="validation",
        download=True,
        **cfg,
    )
    return val1, val2

In [21]:
def check_intra_dataset_determinism(ds, indices=(0, 1, 2, 10, 100)):
    print("Checking intra-dataset determinism...")
    for idx in indices:
        w1, y1, m1 = ds[idx]
        w2, y2, m2 = ds[idx]

        assert y1 == y2, f"Label mismatch at idx={idx}"
        assert torch.allclose(w1, w2), f"Waveform mismatch at idx={idx}"
        
        # Check some meta fields that should be stable
        for key in ["label", "speaker_id", "utterance", "snr"]:
            assert m1[key] == m2[key], f"Meta[{key}] mismatch at idx={idx}"

        # Noise tensor should also be deterministic in eval+add_noise=True
        # (and zero in eval+add_noise=False)
        assert torch.allclose(m1["noise"], m2["noise"]), f"Noise mismatch at idx={idx}"
        
    print("✓ Intra-dataset determinism OK.")


In [22]:
def check_cross_dataset_determinism(ds1, ds2, indices=(0, 1, 2, 10, 100)):
    print("Checking cross-dataset determinism...")
    for idx in indices:
        w1, y1, m1 = ds1[idx]
        w2, y2, m2 = ds2[idx]

        assert y1 == y2, f"Label mismatch at idx={idx}"
        assert torch.allclose(w1, w2), f"Waveform mismatch at idx={idx}"

        for key in ["label", "speaker_id", "utterance", "snr"]:
            assert m1[key] == m2[key], f"Meta[{key}] mismatch at idx={idx}"

        assert torch.allclose(m1["noise"], m2["noise"]), f"Noise mismatch at idx={idx}"

    print("✓ Cross-dataset determinism OK.")


In [23]:
def find_silence_indices(ds, max_search=500):
    idxs = []
    for i in range(min(max_search, len(ds))):
        _, _, meta = ds[i]
        if meta["label"] == "SILENCE":
            idxs.append(i)
    return idxs

def check_silence_determinism(ds1, ds2):
    silence_idxs = find_silence_indices(ds1)
    print(f"Found {len(silence_idxs)} silence indices (showing up to 5): {silence_idxs[:5]}")
    if not silence_idxs:
        print("No silence samples found in this dataset.")
        return

    for idx in silence_idxs[:5]:
        w1, y1, m1 = ds1[idx]
        w2, y2, m2 = ds2[idx]

        assert y1 == y2 == ds1.label_to_idx["SILENCE"], f"Silence label mismatch at idx={idx}"
        assert torch.allclose(w1, w2), f"Silence waveform mismatch at idx={idx}"
        assert m1["label"] == m2["label"] == "SILENCE"

        # In eval:
        # - if add_noise=False -> noise should be zeros
        # - if add_noise=True  -> noise is deterministic noise chunk
        print(f"idx={idx}, snr={m1['snr']}, noise_norm={m1['noise'].norm().item():.4f}")


In [24]:
data_dir = get_data_dir()
val1, val2 = make_val_sets(data_dir, noise_eval_cfg)

check_intra_dataset_determinism(val1)
check_cross_dataset_determinism(val1, val2)
check_silence_determinism(val1, val2)


Checking intra-dataset determinism...
✓ Intra-dataset determinism OK.
Checking cross-dataset determinism...
✓ Cross-dataset determinism OK.
Found 0 silence indices (showing up to 5): []
No silence samples found in this dataset.
